## Celda 1: Configuración de Conexión JDBC

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, current_date, sum as spark_sum, count, avg

# Configuración de conexión PostgreSQL
jdbc_url = "jdbc:postgresql://localhost:5432/electiva"
connection_properties = {
    "user": "postgres",
    "password": "123abc",
    "driver": "org.postgresql.Driver"
}

print("Configuración JDBC definida:")
print(f"URL: {jdbc_url}")
print(f"Driver: {connection_properties['driver']}")

Configuración JDBC definida:
URL: jdbc:postgresql://localhost:5432/electiva
Driver: org.postgresql.Driver


## 2. Celda 2: Crear SparkSession con soporte JDBC

In [2]:
spark = (SparkSession.builder
    .appName("ETL_PostgreSQL")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1500m")
    .config("spark.jars", "/usr/local/spark-3.5.8/jars/postgresql-42.7.3.jar")
    .getOrCreate())

print("SparkSession creada con soporte JDBC")
print(f"App: {spark.sparkContext.appName}")
print(f"Version: {spark.version}")

26/06/24 20:36:20 WARN Utils: Your hostname, david-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 172.16.179.130 instead (on interface ens33)
26/06/24 20:36:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


26/06/24 20:36:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada con soporte JDBC
App: ETL_PostgreSQL
Version: 3.5.8


## 3. Celda 3: Leer Datos desde PostgreSQL

In [3]:
print("=" * 60)
print("LECTURA DESDE POSTGRESQL")
print("=" * 60)

# Leer tabla clientes
df_clientes = spark.read.jdbc(
    url=jdbc_url,
    table="clientes",
    properties=connection_properties
)

print("Tabla 'clientes':")
df_clientes.printSchema()
df_clientes.show()

# Leer tabla ventas
df_ventas = spark.read.jdbc(
    url=jdbc_url,
    table="ventas",
    properties=connection_properties
)

print("\nTabla 'ventas':")
df_ventas.printSchema()
df_ventas.show()

LECTURA DESDE POSTGRESQL


Tabla 'clientes':
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- email: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- fecha_registro: date (nullable = true)
 |-- segmento: string (nullable = true)



+----------+-------------+--------------------+------------+--------------+--------+
|id_cliente|       nombre|               email|      ciudad|fecha_registro|segmento|
+----------+-------------+--------------------+------------+--------------+--------+
|         1|   Ana García|ana.garcia@email.com|      Bogotá|    2023-01-15| Premium|
|         2|Luis Martínez|    luis.m@email.com|    Medellín|    2023-03-22|Estándar|
|         3|  Carla Rojas|   carla.r@email.com|        Cali|    2023-06-10| Premium|
|         4|Pedro Sánchez|   pedro.s@email.com|Barranquilla|    2023-08-05|  Básico|
|         5|  María López|   maria.l@email.com|      Bogotá|    2023-09-12|Estándar|
+----------+-------------+--------------------+------------+--------------+--------+


Tabla 'ventas':
root
 |-- id_venta: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(10,2) (nullable =

+--------+----------+-------------------+--------+---------------+-----------+-------+
|id_venta|id_cliente|           producto|cantidad|precio_unitario|fecha_venta|  total|
+--------+----------+-------------------+--------+---------------+-----------+-------+
|       1|         1|         Laptop Pro|       1|        2500.00| 2024-01-10|2500.00|
|       2|         1|  Mouse Inalámbrico|       2|          25.50| 2024-01-15|  51.00|
|       3|         2|   Teclado Mecánico|       1|         120.00| 2024-02-20| 120.00|
|       4|         3|         Monitor 4K|       2|         450.00| 2024-03-05| 900.00|
|       5|         3|          Webcam HD|       1|          80.00| 2024-03-10|  80.00|
|       6|         4|Audífonos Bluetooth|       3|          60.00| 2024-04-01| 180.00|
|       7|         5| Disco Duro Externo|       1|         150.00| 2024-04-15| 150.00|
+--------+----------+-------------------+--------+---------------+-----------+-------+



## 4. Celda 4: Transformaciones con DataFrame

In [4]:
print("=" * 60)
print("TRANSFORMACIONES")
print("=" * 60)

# Join entre clientes y ventas
df_joined = df_clientes.join(
    df_ventas,
    df_clientes.id_cliente == df_ventas.id_cliente,
    "inner"
).select(
    df_clientes.id_cliente,
    "nombre",
    "ciudad",
    "segmento",
    "producto",
    "cantidad",
    "precio_unitario",
    "total",
    "fecha_venta"
)
print("Join Clientes + Ventas:")
df_joined.show()
# Agregaciones por segmento
df_por_segmento = df_joined.groupBy("segmento").agg(
    count("*").alias("total_ventas"),
    spark_sum("total").alias("ingresos_totales"),
    avg("total").alias("ticket_promedio")
)
print("\nAgregación por segmento:")
df_por_segmento.show()
# Clientes Premium con alto valor
df_premium = df_joined.filter(
    (col("segmento") == "Premium") & (col("total") > 100)
).withColumn(
    "categoria_valor",
    when(col("total") > 1000, "Alto Valor")
    .when(col("total") > 500, "Medio Valor")
    .otherwise("Bajo Valor")
)
print("\nClientes Premium con categorización:")
df_premium.show()

TRANSFORMACIONES


Join Clientes + Ventas:


+----------+-------------+------------+--------+-------------------+--------+---------------+-------+-----------+
|id_cliente|       nombre|      ciudad|segmento|           producto|cantidad|precio_unitario|  total|fecha_venta|
+----------+-------------+------------+--------+-------------------+--------+---------------+-------+-----------+
|         1|   Ana García|      Bogotá| Premium|         Laptop Pro|       1|        2500.00|2500.00| 2024-01-10|
|         1|   Ana García|      Bogotá| Premium|  Mouse Inalámbrico|       2|          25.50|  51.00| 2024-01-15|
|         2|Luis Martínez|    Medellín|Estándar|   Teclado Mecánico|       1|         120.00| 120.00| 2024-02-20|
|         3|  Carla Rojas|        Cali| Premium|         Monitor 4K|       2|         450.00| 900.00| 2024-03-05|
|         3|  Carla Rojas|        Cali| Premium|          Webcam HD|       1|          80.00|  80.00| 2024-03-10|
|         4|Pedro Sánchez|Barranquilla|  Básico|Audífonos Bluetooth|       3|          6

+--------+------------+----------------+---------------+
|segmento|total_ventas|ingresos_totales|ticket_promedio|
+--------+------------+----------------+---------------+
| Premium|           4|         3531.00|     882.750000|
|Estándar|           2|          270.00|     135.000000|
|  Básico|           1|          180.00|     180.000000|
+--------+------------+----------------+---------------+


Clientes Premium con categorización:


+----------+-----------+------+--------+----------+--------+---------------+-------+-----------+---------------+
|id_cliente|     nombre|ciudad|segmento|  producto|cantidad|precio_unitario|  total|fecha_venta|categoria_valor|
+----------+-----------+------+--------+----------+--------+---------------+-------+-----------+---------------+
|         1| Ana García|Bogotá| Premium|Laptop Pro|       1|        2500.00|2500.00| 2024-01-10|     Alto Valor|
|         3|Carla Rojas|  Cali| Premium|Monitor 4K|       2|         450.00| 900.00| 2024-03-05|    Medio Valor|
+----------+-----------+------+--------+----------+--------+---------------+-------+-----------+---------------+



## 5. Celda 5: Escritura a PostgreSQL (Modo Append)

In [5]:
from pyspark.sql.types import DateType

print("=" * 60)
print("ESCRITURA A POSTGRESQL (APPEND)")
print("=" * 60)

# Crear tabla resumen de ventas por cliente
df_resumen = df_joined.groupBy("id_cliente", "nombre", "ciudad", "segmento").agg(
    count("*").alias("num_compras"),
    spark_sum("total").alias("total_gastado"),
    avg("total").alias("promedio_compra")
).withColumn("fecha_analisis", current_date().cast(DateType()))

print("DataFrame a escribir (resumen_clientes):")
df_resumen.printSchema()
df_resumen.show()

# Escribir en PostgreSQL (modo overwrite para crear tabla)
df_resumen.write.jdbc(
    url=jdbc_url,
    table="resumen_clientes",
    mode="overwrite",  # Crea o reemplaza la tabla
    properties=connection_properties
)
print("Tabla 'resumen_clientes' creada en PostgreSQL")

ESCRITURA A POSTGRESQL (APPEND)


DataFrame a escribir (resumen_clientes):
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- num_compras: long (nullable = false)
 |-- total_gastado: decimal(20,2) (nullable = true)
 |-- promedio_compra: decimal(14,6) (nullable = true)
 |-- fecha_analisis: date (nullable = false)



+----------+-------------+------------+--------+-----------+-------------+---------------+--------------+
|id_cliente|       nombre|      ciudad|segmento|num_compras|total_gastado|promedio_compra|fecha_analisis|
+----------+-------------+------------+--------+-----------+-------------+---------------+--------------+
|         1|   Ana García|      Bogotá| Premium|          2|      2551.00|    1275.500000|    2026-06-24|
|         2|Luis Martínez|    Medellín|Estándar|          1|       120.00|     120.000000|    2026-06-24|
|         3|  Carla Rojas|        Cali| Premium|          2|       980.00|     490.000000|    2026-06-24|
|         4|Pedro Sánchez|Barranquilla|  Básico|          1|       180.00|     180.000000|    2026-06-24|
|         5|  María López|      Bogotá|Estándar|          1|       150.00|     150.000000|    2026-06-24|
+----------+-------------+------------+--------+-----------+-------------+---------------+--------------+



Tabla 'resumen_clientes' creada en PostgreSQL


## 6. Celda 6: Verificar Escritura y Lectura de Vuelta

In [6]:
print("=" * 60)
print("APPEND DE DATOS ADICIONALES (CON DATE EXPLÍCITO)")
print("=" * 60)
from datetime import date

# Crear DataFrame con fecha como DateType (NO como string)
from pyspark.sql import Row

# Usar Row con fecha como objeto date de Python
df_append = spark.createDataFrame([
    Row(
        id_cliente=99,
        nombre="Cliente Nuevo",
        ciudad="Cartagena",
        segmento="Estándar",
        num_compras=2,
        total_gastado=300.00,
        promedio_compra=150.00,
        fecha_analisis=date(2024, 5, 20) # ← date de Python, no string
    )
])

# Verificar esquema antes de escribir
print("Esquema del DataFrame a insertar:")
df_append.printSchema()
df_append.show()

# Escribir en modo append
df_append.write.jdbc(
    url=jdbc_url,
    table="resumen_clientes",
    mode="append",
    properties=connection_properties
)
print("✅ Datos adicionales insertados en modo APPEND")

# Verificar
df_final = spark.read.jdbc(
    url=jdbc_url,
    table="resumen_clientes",
    properties=connection_properties
)

print("\nTabla final con datos append:")
df_final.show()
df_final.printSchema()

APPEND DE DATOS ADICIONALES (CON DATE EXPLÍCITO)


Esquema del DataFrame a insertar:
root
 |-- id_cliente: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- num_compras: long (nullable = true)
 |-- total_gastado: double (nullable = true)
 |-- promedio_compra: double (nullable = true)
 |-- fecha_analisis: date (nullable = true)



+----------+-------------+---------+--------+-----------+-------------+---------------+--------------+
|id_cliente|       nombre|   ciudad|segmento|num_compras|total_gastado|promedio_compra|fecha_analisis|
+----------+-------------+---------+--------+-----------+-------------+---------------+--------------+
|        99|Cliente Nuevo|Cartagena|Estándar|          2|        300.0|          150.0|    2024-05-20|
+----------+-------------+---------+--------+-----------+-------------+---------------+--------------+



✅ Datos adicionales insertados en modo APPEND

Tabla final con datos append:


+----------+-------------+------------+--------+-----------+-------------+---------------+--------------+
|id_cliente|       nombre|      ciudad|segmento|num_compras|total_gastado|promedio_compra|fecha_analisis|
+----------+-------------+------------+--------+-----------+-------------+---------------+--------------+
|         1|   Ana García|      Bogotá| Premium|          2|      2551.00|    1275.500000|    2026-06-24|
|         2|Luis Martínez|    Medellín|Estándar|          1|       120.00|     120.000000|    2026-06-24|
|         3|  Carla Rojas|        Cali| Premium|          2|       980.00|     490.000000|    2026-06-24|
|         4|Pedro Sánchez|Barranquilla|  Básico|          1|       180.00|     180.000000|    2026-06-24|
|         5|  María López|      Bogotá|Estándar|          1|       150.00|     150.000000|    2026-06-24|
|        99|Cliente Nuevo|   Cartagena|Estándar|          2|       300.00|     150.000000|    2024-05-20|
+----------+-------------+------------+-------

## 7. Manejo de Tipos de Datos (DATE)

In [7]:
print("=" * 60)
print("MANEJO DE TIPOS DE DATOS - DATE EN POSTGRESQL")
print("=" * 60)

# Verificar tipos de fecha en el DataFrame original
print("Esquema de ventas (tipos originales):")
df_ventas.printSchema()

# Convertir fecha_venta a DateType explícito
df_ventas_fecha = df_ventas.withColumn("fecha_venta",
col("fecha_venta").cast(DateType()))
print("\nEsquema con fecha convertida:")
df_ventas_fecha.printSchema()

# Filtrar por rango de fechas
df_marzo = df_ventas_fecha.filter(
(col("fecha_venta") >= "2024-03-01") &
(col("fecha_venta") <= "2024-03-31")
)
print("\nVentas de marzo 2024:")
df_marzo.show()

# Escribir particionado por fecha (sobrescribir tabla de prueba)
df_ventas_fecha.write \
.option("createTableColumnTypes", "fecha_venta DATE") \
.jdbc(url=jdbc_url, table="ventas_particionadas", mode="overwrite",
properties=connection_properties)
print("✅ Datos escritos con tipo DATE explícito en PostgreSQL")

# Verificar tipos en PostgreSQL
df_verificacion_pg = spark.read.jdbc(
url=jdbc_url,
table="ventas_particionadas",
properties=connection_properties
)
print("\nEsquema leído desde PostgreSQL:")
df_verificacion_pg.printSchema()

MANEJO DE TIPOS DE DATOS - DATE EN POSTGRESQL
Esquema de ventas (tipos originales):
root
 |-- id_venta: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(10,2) (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- total: decimal(10,2) (nullable = true)


Esquema con fecha convertida:
root
 |-- id_venta: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(10,2) (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- total: decimal(10,2) (nullable = true)


Ventas de marzo 2024:


+--------+----------+----------+--------+---------------+-----------+------+
|id_venta|id_cliente|  producto|cantidad|precio_unitario|fecha_venta| total|
+--------+----------+----------+--------+---------------+-----------+------+
|       4|         3|Monitor 4K|       2|         450.00| 2024-03-05|900.00|
|       5|         3| Webcam HD|       1|          80.00| 2024-03-10| 80.00|
+--------+----------+----------+--------+---------------+-----------+------+



✅ Datos escritos con tipo DATE explícito en PostgreSQL

Esquema leído desde PostgreSQL:
root
 |-- id_venta: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(10,2) (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- total: decimal(10,2) (nullable = true)

